<a href="https://colab.research.google.com/github/shahzamanjatoi/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzamanjatoi/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook builds a transparent, hand-written baseline score to prioritize pages for review using February 2026 search-performance signals.

The analysis uses:

- **February 2026** as the feature window.
- **March 2026** as the future evaluation window.
- **CTR gap** relative to the median CTR within each position tier.
- **Search volume** based on February impressions.
- **`went_dark`** as the March outcome label.

The goal is to test whether these observable signals provide useful directional information for prioritizing pages for review.

The baseline is intentionally simple and interpretable. The March outcome is used only for evaluation and is not used to calculate the ranking score.

> **Interpretation:** Results are treated as observed and directional decision-support, not as claims about Google's ranking algorithm.

In [21]:
!pip install -q huggingface_hub

from huggingface_hub import login

login()

In [22]:
import os
import getpass
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# -----------------------------
# Connect to the FlyRank warehouse
# -----------------------------
def get_hf_token():
    token = userdata.get("HF_TOKEN")
    if token:
        return token

    return getpass.getpass(
        "Enter your Hugging Face READ token (hf_...): "
    )

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

token = get_hf_token()

# Keep the token out of the SQL text.
con.execute("SET VARIABLE hf_token = ?", [token])
con.execute("""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN getvariable('hf_token'))
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

# February = feature window
FEB = f"{FACT}/month=2026-02/*.parquet"

# March = future evaluation/label window
MAR = f"{FACT}/month=2026-03/*.parquet"

DIM = f"{REL}/dim_content.parquet"

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Label window: March 2026")

Connected to FlyRank warehouse.
Feature window: February 2026
Label window: March 2026


In [23]:
# ---------------------------------------
# Build February features
# ---------------------------------------

feb = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS impressions_feb,

        SUM(gsc_clicks)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS clicks_feb,

        SUM(gsc_sum_position)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS sum_position_feb,

        COUNT(*)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS measured_days_feb

    FROM read_parquet('{FEB}')
    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING
        SUM(gsc_impressions)
            FILTER (WHERE gsc_data_available IS TRUE) >= 100

        AND

        SUM(gsc_clicks)
            FILTER (WHERE gsc_data_available IS TRUE) >= 3
""").df()

feb["ctr_feb"] = (
    feb["clicks_feb"] /
    feb["impressions_feb"] *
    100
)

feb["avg_position_feb"] = (
    feb["sum_position_feb"] /
    feb["impressions_feb"]
)

# ---------------------------------------
# Position tier
# ---------------------------------------

feb["position_tier"] = pd.cut(
    feb["avg_position_feb"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=[
        "top_3",
        "page_1",
        "striking",
        "page_3_5",
        "deep"
    ],
    include_lowest=True
)

print("February feature rows:", len(feb))
print()
print(feb[
    [
        "client_hash_id",
        "content_hash_id",
        "impressions_feb",
        "clicks_feb",
        "ctr_feb",
        "avg_position_feb",
        "position_tier"
    ]
].head())

February feature rows: 29729

            client_hash_id           content_hash_id  impressions_feb  \
0  client_e547b89c05043229  content_9abd8b303f805847            733.0   
1  client_e547b89c05043229  content_6fe390ba3af1e456           2931.0   
2  client_e547b89c05043229  content_babd931911c9ee33           2680.0   
3  client_e547b89c05043229  content_431784c057b25a5d           3641.0   
4  client_e547b89c05043229  content_7275e583711cf60b           2371.0   

   clicks_feb   ctr_feb  avg_position_feb position_tier  
0         6.0  0.818554          6.316508        page_1  
1         3.0  0.102354         41.814739      page_3_5  
2        31.0  1.156716          5.049254        page_1  
3         6.0  0.164790          8.829992        page_1  
4         6.0  0.253058          7.358077        page_1  


In [24]:
# ---------------------------------------
# Build March future outcome
# ---------------------------------------

mar = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS impressions_mar,

        SUM(gsc_clicks)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS clicks_mar,

        COUNT(*)
            FILTER (WHERE gsc_data_available IS TRUE)
            AS measured_days_mar

    FROM read_parquet('{MAR}')
    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

# Keep only pages with at least one measured March day.
mar = mar[mar["measured_days_mar"] > 0].copy()

mar["impressions_mar"] = mar["impressions_mar"].fillna(0)
mar["clicks_mar"] = mar["clicks_mar"].fillna(0)

# Future label:
# 1 = zero clicks in March
# 0 = at least one click in March
mar["went_dark"] = (
    mar["clicks_mar"] == 0
).astype(int)

# ---------------------------------------
# Merge February features with March label
# ---------------------------------------

frame = feb.merge(
    mar[
        [
            "client_hash_id",
            "content_hash_id",
            "impressions_mar",
            "clicks_mar",
            "measured_days_mar",
            "went_dark"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Final baseline frame:", frame.shape)
print()
print("Future-label base rate:")
print(frame["went_dark"].mean())

print()
print(frame["went_dark"].value_counts())

Final baseline frame: (29368, 13)

Future-label base rate:
0.03983928084990466

went_dark
0    28198
1     1170
Name: count, dtype: int64


## 1. My rule and its reason codes

### My baseline rule

I will rank pages for review using two observable February signals:

1. **CTR gap:** whether a page's CTR is below the median CTR for pages in the same position tier.
2. **Search volume:** whether the page has at least 100 February impressions.

The hand-written score is:

- +2 points if the page has a negative CTR gap and at least 100 impressions.
- +1 point if the page has at least 100 impressions.

Higher scores receive higher review priority.

This is an explicit, transparent baseline rule. The March outcome is used only to evaluate whether the signals are useful; it is not used to calculate the score.

This is directional decision-support, not a claim about Google's ranking algorithm.

In [25]:
# ---------------------------------------
# Signal definitions
# ---------------------------------------

# Position-tier CTR benchmark
tier_ctr = (
    frame
    .groupby("position_tier", observed=True)["ctr_feb"]
    .median()
    .rename("position_tier_median_ctr")
)

frame = frame.join(
    tier_ctr,
    on="position_tier"
)

# CTR gap:
# negative = below the typical CTR for its position tier
frame["ctr_gap"] = (
    frame["ctr_feb"] -
    frame["position_tier_median_ctr"]
)

# Signal 1: CTR opportunity
frame["ctr_fix_signal"] = (
    (frame["ctr_gap"] < 0) &
    (frame["impressions_feb"] >= 100)
)

# Signal 2: volume
frame["volume_signal"] = (
    frame["impressions_feb"] >= 100
)

# Transparent score
frame["score"] = (
    2 * frame["ctr_fix_signal"].astype(int)
    + frame["volume_signal"].astype(int)
)

# Reason codes
frame["reason_code"] = np.select(
    [
        frame["ctr_fix_signal"] & frame["volume_signal"],
        frame["ctr_fix_signal"],
        frame["volume_signal"]
    ],
    [
        "ctr_fix_high_volume",
        "ctr_gap_only",
        "volume_only"
    ],
    default="no_signal"
)

# Action labels
frame["action"] = np.select(
    [
        frame["ctr_fix_signal"],
        frame["volume_signal"]
    ],
    [
        "CTR_FIX",
        "MONITOR"
    ],
    default="NO_ACTION"
)

print(
    frame[
        [
            "impressions_feb",
            "ctr_feb",
            "position_tier",
            "position_tier_median_ctr",
            "ctr_gap",
            "score",
            "reason_code",
            "action"
        ]
    ].head(10)
)

   impressions_feb   ctr_feb position_tier  position_tier_median_ctr  \
0            733.0  0.818554        page_1                  0.379867   
1           2931.0  0.102354      page_3_5                  0.209644   
2           2680.0  1.156716        page_1                  0.379867   
3           3641.0  0.164790        page_1                  0.379867   
4           2371.0  0.253058        page_1                  0.379867   
5            798.0  0.375940      striking                  0.408094   
6           1351.0  0.222058        page_1                  0.379867   
7           3940.0  0.152284      page_3_5                  0.209644   
8            676.0  0.443787      striking                  0.408094   
9           4917.0  0.386414        page_1                  0.379867   

    ctr_gap  score          reason_code   action  
0  0.438687      1          volume_only  MONITOR  
1 -0.107289      3  ctr_fix_high_volume  CTR_FIX  
2  0.776849      1          volume_only  MONITOR  
3 -

In [26]:
# ---------------------------------------
# Signal test 1: CTR gap
# ---------------------------------------

frame["ctr_gap_bucket"] = pd.qcut(
    frame["ctr_gap"],
    q=4,
    labels=[
        "Q1_low",
        "Q2",
        "Q3",
        "Q4_high"
    ],
    duplicates="drop"
)

ctr_test = (
    frame
    .groupby("ctr_gap_bucket", observed=True)
    .agg(
        n=("went_dark", "size"),
        went_dark_rate=("went_dark", "mean"),
        median_ctr_gap=("ctr_gap", "median")
    )
    .reset_index()
)

print("SIGNAL 1 — CTR gap")
print(ctr_test.to_string(index=False))

# ---------------------------------------
# Signal test 2: search volume
# ---------------------------------------

frame["volume_bucket"] = pd.cut(
    frame["impressions_feb"],
    bins=[0, 100, 500, 3000, np.inf],
    labels=[
        "100-499",
        "500-2999",
        "3000-29999",
        "30000+"
    ],
    include_lowest=True
)

volume_test = (
    frame
    .groupby("volume_bucket", observed=True)
    .agg(
        n=("went_dark", "size"),
        went_dark_rate=("went_dark", "mean"),
        median_impressions=("impressions_feb", "median")
    )
    .reset_index()
)

print()
print("SIGNAL 2 — Search volume")
print(volume_test.to_string(index=False))

SIGNAL 1 — CTR gap
ctr_gap_bucket    n  went_dark_rate  median_ctr_gap
        Q1_low 7342        0.035413       -0.225682
            Q2 7346        0.038252       -0.083638
            Q3 7338        0.032025        0.082036
       Q4_high 7342        0.053664        0.549470

SIGNAL 2 — Search volume
volume_bucket     n  went_dark_rate  median_impressions
      100-499     3        0.333333               100.0
     500-2999  1779        0.155705               333.0
   3000-29999 14460        0.051314              1607.5
       30000+ 13126        0.011428              5923.0


## Signal verdicts

### Signal 1 — CTR gap

**Verdict: OPPOSITE**

The CTR-gap buckets do not support the baseline assumption that a more negative CTR gap leads to a higher March `went_dark` rate.

The lowest CTR-gap bucket has a `went_dark` rate of about 3.54%, while the highest CTR-gap bucket has about 5.37%. Therefore, the observed relationship is not monotonic in the expected direction.

This signal should not be described as confirmed evidence. It is better treated as an opposite/misaligned signal for this outcome.

### Signal 2 — Search volume

**Verdict: CONFIRMED**

Search volume shows a clear monotonic relationship with the March outcome.

The `went_dark` rate decreases from approximately 33.33% in the 100–499 bucket to 1.14% in the 30,000+ bucket.

Therefore, higher February search volume is associated with a lower probability of going dark in March.

This confirms that search volume contains useful directional information for the observed outcome, although the relationship is inverse to the assumption that high-volume pages should automatically receive higher risk priority.

In [27]:
# ---------------------------------------
# Explicit signal verdicts
# ---------------------------------------

signal_verdicts = pd.DataFrame({
    "signal": [
        "CTR gap",
        "Search volume"
    ],
    "verdict": [
        "OPPOSITE",
        "CONFIRMED"
    ],
    "reason": [
        "The CTR-gap buckets do not increase in the expected direction.",
        "Went-dark rate decreases consistently as search volume increases."
    ]
})

print(signal_verdicts.to_string(index=False))

       signal   verdict                                                            reason
      CTR gap  OPPOSITE    The CTR-gap buckets do not increase in the expected direction.
Search volume CONFIRMED Went-dark rate decreases consistently as search volume increases.


## 2. Build the ranked queue

The baseline score is deliberately hand-written rather than fitted.

A page receives:

- 2 points for the CTR-fix signal;
- 1 point for sufficient search volume.

The queue is sorted from highest score to lowest score.

The score is an operational review-priority rule, not a claim that higher scores predict `went_dark`.

The signal audit shows that the CTR-gap signal is opposite to the expected direction and search volume is confirmed but inverse to the assumed risk relationship. Therefore, this baseline should be treated as a transparent first-pass review rule rather than a validated predictive model.

The March outcome is used only for evaluation and is not used to calculate the score.

In [28]:
# ---------------------------------------
# Build ranked queue
# ---------------------------------------

queue = frame.copy()

queue = queue.sort_values(
    [
        "score",
        "ctr_gap",
        "impressions_feb"
    ],
    ascending=[
        False,
        True,
        False
    ]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

queue = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "impressions_feb",
        "ctr_feb",
        "avg_position_feb",
        "position_tier",
        "position_tier_median_ctr",
        "ctr_gap",
        "score",
        "reason_code",
        "action",
        "went_dark"
    ]
]

# ---------------------------------------
# Precision@K
# ---------------------------------------

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = queue["went_dark"].mean()

for k in [10, 20, 50]:
    p_at_k = precision_at_k(
        queue["score"],
        queue["went_dark"],
        k
    )

    print(
        f"Precision@{k}: {p_at_k:.3f}"
    )

print(f"Base rate: {base_rate:.3f}")

# ---------------------------------------
# Write required CSV
# ---------------------------------------

output_path = "work/outputs/baseline_action_score.csv"

os.makedirs(
    "work/outputs",
    exist_ok=True
)

queue.drop(
    columns=["went_dark"]
).to_csv(
    output_path,
    index=False
)

print()
print("Wrote:", output_path)
print("Rows:", len(queue))

print()
print(queue.head(10).to_string(index=False))

Precision@10: 0.000
Precision@20: 0.000
Precision@50: 0.080
Base rate: 0.040

Wrote: work/outputs/baseline_action_score.csv
Rows: 29368

 rank          client_hash_id          content_hash_id  impressions_feb  ctr_feb  avg_position_feb position_tier  position_tier_median_ctr   ctr_gap  score         reason_code  action  went_dark
    1 client_23a62021009f63c4 content_dc876cd35c8f5e94          24699.0 0.016195         18.727438      striking                  0.408094 -0.391899      3 ctr_fix_high_volume CTR_FIX          0
    2 client_23a62021009f63c4 content_601538d3db57a478          15711.0 0.019095         12.045510      striking                  0.408094 -0.389000      3 ctr_fix_high_volume CTR_FIX          0
    3 client_23a62021009f63c4 content_7daf3162408376d2          26703.0 0.022469         16.312474      striking                  0.408094 -0.385625      3 ctr_fix_high_volume CTR_FIX          0
    4 client_23a62021009f63c4 content_bd63db2d0757e760          17327.0 0.023085   

### Baseline evaluation

The observed base rate of `went_dark` is approximately 4.0%.

The baseline achieves:

- Precision@10 = 0.0%
- Precision@20 = 0.0%
- Precision@50 = 8.0%

Therefore, Precision@50 is approximately 2× the base rate, but the ranking performs poorly at the smallest review budgets.

This means the baseline provides some ranking lift at K=50, but the evidence is not strong enough to treat the rule as a reliable predictive model.

The result should be interpreted as directional decision-support and as a baseline for comparison with later models.

## 3. Top-10 review

I review the top ten rows rather than treating the score as automatically correct.

For every row I record:
- the action;
- the reason code;
- a confidence note;
- and what could make the recommendation wrong.

The confidence note reflects the amount and consistency of observable evidence, not a probability produced by a trained model.

In [29]:
# ---------------------------------------
# Top-10 review
# ---------------------------------------

top10 = queue.head(10).copy()

def confidence_note(row):
    if (
        row["action"] == "CTR_FIX"
        and row["impressions_feb"] >= 3000
        and row["ctr_gap"] < 0
    ):
        return "Higher confidence: strong volume and negative CTR gap."

    if (
        row["action"] == "CTR_FIX"
        and row["impressions_feb"] >= 100
    ):
        return "Medium confidence: CTR gap is present with minimum volume."

    if row["action"] == "MONITOR":
        return "Lower confidence: volume is present but CTR evidence is weak."

    return "Low confidence: limited supporting signal."


def wrong_if(row):
    if row["action"] == "CTR_FIX":
        return (
            "Could be wrong if SERP features, query intent, or "
            "position-tier differences explain the lower CTR."
        )

    if row["action"] == "MONITOR":
        return (
            "Could be wrong if high impressions do not translate "
            "into a meaningful review opportunity."
        )

    return (
        "Could be wrong if the available signals do not capture "
        "the page's actual search opportunity."
    )


top10["confidence_note"] = top10.apply(
    confidence_note,
    axis=1
)

top10["what_would_make_it_wrong"] = top10.apply(
    wrong_if,
    axis=1
)

review_columns = [
    "rank",
    "action",
    "reason_code",
    "score",
    "impressions_feb",
    "ctr_feb",
    "avg_position_feb",
    "ctr_gap",
    "confidence_note",
    "what_would_make_it_wrong"
]

print(
    top10[review_columns].to_string(index=False)
)

 rank  action         reason_code  score  impressions_feb  ctr_feb  avg_position_feb   ctr_gap                                        confidence_note                                                                           what_would_make_it_wrong
    1 CTR_FIX ctr_fix_high_volume      3          24699.0 0.016195         18.727438 -0.391899 Higher confidence: strong volume and negative CTR gap. Could be wrong if SERP features, query intent, or position-tier differences explain the lower CTR.
    2 CTR_FIX ctr_fix_high_volume      3          15711.0 0.019095         12.045510 -0.389000 Higher confidence: strong volume and negative CTR gap. Could be wrong if SERP features, query intent, or position-tier differences explain the lower CTR.
    3 CTR_FIX ctr_fix_high_volume      3          26703.0 0.022469         16.312474 -0.385625 Higher confidence: strong volume and negative CTR gap. Could be wrong if SERP features, query intent, or position-tier differences explain the lower CTR.
    

## 4. Weak picks + leakage check

The weakest picks are pages where the rule can be triggered by a simple threshold without enough context to establish that the recommended action is truly useful.

The main weak-pick risks are:
- low-volume CTR instability;
- SERP features or query-intent differences that are not represented in the rule;
- pages where high impressions create a large queue priority without a clear content opportunity.

### Leakage check

The score uses only February 2026 observable signals:
`impressions_feb`, `clicks_feb`, `ctr_feb`, and February position.

The March `went_dark` outcome is used only after ranking for evaluation. It is not used to calculate the score, reason code, or action.

No FlyRank product flag, future-window feature, `trend_direction`, or `trend_pct` is used in the baseline.

In [30]:
# ---------------------------------------
# Weak-pick review
# ---------------------------------------

weak_picks = queue[
    (
        (queue["action"] == "MONITOR") |
        (queue["impressions_feb"] < 500)
    )
].head(5)

print("Potential weak picks:")
print(
    weak_picks[
        [
            "rank",
            "action",
            "reason_code",
            "score",
            "impressions_feb",
            "ctr_feb",
            "ctr_gap"
        ]
    ].to_string(index=False)
)

# ---------------------------------------
# Leakage assertions
# ---------------------------------------

score_columns = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
    "avg_position_feb",
    "position_tier",
    "position_tier_median_ctr",
    "ctr_gap"
]

for col in score_columns:
    assert col in frame.columns, f"Missing expected feature: {col}"

for forbidden in [
    "trend_direction",
    "trend_pct",
    "went_dark"
]:
    assert forbidden not in score_columns, (
        f"Potential leakage: {forbidden}"
    )

print()
print("Leakage checks passed.")
print("March outcome was used only for evaluation.")
print("No future-derived feature was used in the score.")

Potential weak picks:
 rank  action reason_code  score  impressions_feb  ctr_feb  ctr_gap
14683 MONITOR volume_only      1          11746.0 0.102162      0.0
14684 MONITOR volume_only      1           4960.0 0.383065      0.0
14685 MONITOR volume_only      1           3159.0 0.379867      0.0
14686 MONITOR volume_only      1           1431.0 0.209644      0.0
14687 MONITOR volume_only      1           1053.0 0.379867      0.0

Leakage checks passed.
March outcome was used only for evaluation.
No future-derived feature was used in the score.


## 5. Final assignment verdict

### Signal findings

| Signal | Verdict | Evidence |
|---|---|---|
| CTR gap | OPPOSITE | The lowest CTR-gap bucket has 3.54% went-dark versus 5.37% in the highest bucket, so the expected direction is not supported. |
| Search volume | CONFIRMED | Went-dark rate decreases from 33.33% in 100–499 impressions to 1.14% in 30,000+ impressions. |

### Explicit baseline rule

Rank pages using:

`score = 2 × CTR_FIX + 1 × VOLUME`

where:

- `CTR_FIX = 1` when CTR is below the median for its position tier and February impressions are at least 100.
- `VOLUME = 1` when February impressions are at least 100.

Sort descending by score, then by more negative CTR gap.

### Baseline performance

The March went-dark base rate is approximately 4.0%.

The baseline achieves:

- Precision@10 = 0.0%
- Precision@20 = 0.0%
- Precision@50 = 8.0%

Therefore, the baseline provides some lift at K=50 but is not reliable at very small review budgets.

### Overall conclusion

The signal audit does not support the original assumption that a negative CTR gap predicts future pages going dark. Search volume shows a much clearer inverse relationship with the observed outcome.

The baseline is therefore retained as a transparent operational ranking rule and comparison point, not as a validated predictive model.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.